# Preprocesamiento sin fuga de informacion

Carlos Zambrano Figueroa | Grupo 11

Objetivo: conservar las 18 variables y el preprocesamiento de referencia, ajustando cada transformador solo con su fold de entrenamiento. No se entrenan clasificadores ni se evalua la reserva final.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src/data_processing.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src/data_processing.py').exists():
    raise RuntimeError('Ejecutar desde la raiz del repositorio o notebooks/')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from src.data_processing import (
    CATEGORICAL, FEATURES, NUMERIC, build_preprocessor,
    load_development, select_predictors,
)
from src.preprocessing_audit import run

development, folds = load_development()
X = select_predictors(development)
print('Filas de desarrollo:', len(X))
print('Variables numericas:', len(NUMERIC))
print('Variables categoricas:', len(CATEGORICAL))
print('La reserva final no se evalua.')


Filas de desarrollo: 3539
Variables numericas: 4
Variables categoricas: 14
La reserva final no se evalua.


## 1. Contrato de entrada

Se excluyen Target, indicadores semestrales y variables de temporalidad no confirmada. Los codigos de las variables categoricas se almacenan como numeros pero no se interpretan como magnitudes. La seleccion no aprende estadisticas.


In [2]:
assert list(X.columns) == FEATURES
assert 'Target' not in X.columns
assert 'Scholarship holder' not in X.columns
assert 'Debtor' not in X.columns
display(pd.DataFrame({'variable': FEATURES, 'tratamiento': [
    'mediana + StandardScaler' if c in NUMERIC else 'moda + one-hot'
    for c in FEATURES
]}))


,variable,tratamiento
0,Marital status,moda + one-hot
1,Application mode,moda + one-hot
2,Application order,mediana + StandardScaler
3,Course,moda + one-hot
4,Daytime/evening attendance,moda + one-hot
5,Previous qualification,moda + one-hot
6,Previous qualification (grade),mediana + StandardScaler
7,Nacionality,moda + one-hot
8,Mother's qualification,moda + one-hot
9,Father's qualification,moda + one-hot


## 2. Ajuste exclusivo en entrenamiento

Se ilustra el primer fold. La imputacion usa mediana para numeros y moda para categorias; el escalado aprende media y varianza del entrenamiento. Aunque desarrollo no presenta nulos, se mantiene la imputacion del pipeline historico. No se agrupan categorias por defecto.


In [3]:
train_ids, val_ids = folds[0]
train = X.iloc[train_ids]
validation = X.iloc[val_ids]
preprocessor = build_preprocessor()
train_matrix = preprocessor.fit_transform(train)
validation_matrix = preprocessor.transform(validation)
print('Entrenamiento transformado:', train_matrix.shape)
print('Validacion transformada:', validation_matrix.shape)
assert np.isfinite(train_matrix).all()
assert np.isfinite(validation_matrix).all()

numeric = preprocessor.named_transformers_['numeric']
display(pd.DataFrame({
    'variable': NUMERIC,
    'mediana_entrenamiento': numeric['impute'].statistics_,
    'media_escalador': numeric['scale'].mean_,
    'varianza_escalador': numeric['scale'].var_,
}))


Entrenamiento transformado: (2831, 213)
Validacion transformada: (708, 213)


,variable,mediana_entrenamiento,media_escalador,varianza_escalador
0,Application order,1.0,1.706817,1.652652
1,Previous qualification (grade),133.1,132.766408,171.649486
2,Admission grade,126.1,127.032886,210.097636
3,Age at enrollment,20.0,23.305546,57.365137


## 3. Comprobacion sintetica de estado inmutable

Se modifican valores en una copia de una fila de validacion solo para comprobar el comportamiento del transformador. No son datos reales, no se guardan como dataset y no representan un experimento de rendimiento. Una categoria nueva se codifica con ceros en su bloque; esto no valida su uso institucional.


In [4]:
synthetic = validation.iloc[[0]].copy()
synthetic[NUMERIC] = np.nan
synthetic[CATEGORICAL[0]] = -999
means_before = numeric['scale'].mean_.copy()
known_before = preprocessor.named_transformers_['categorical']['encode'].categories_[0].copy()
transformed_synthetic = preprocessor.transform(synthetic)
np.testing.assert_array_equal(means_before, numeric['scale'].mean_)
np.testing.assert_array_equal(
    known_before,
    preprocessor.named_transformers_['categorical']['encode'].categories_[0],
)
np.testing.assert_array_equal(transformed_synthetic[0, 4:4+len(known_before)], 0.)
assert np.isfinite(transformed_synthetic).all()
print('Faltantes transformados; categoria nueva no incorporada; estado sin cambios.')


Faltantes transformados; categoria nueva no incorporada; estado sin cambios.


## 4. Auditoria de los cinco folds

Cada fold crea un transformador independiente. El numero de columnas codificadas puede variar entre folds, pero debe coincidir entre entrenamiento y validacion del mismo fold. La evidencia incluye las estadisticas aprendidas y los conteos de categorias de validacion no vistas en entrenamiento.


In [5]:
report = run()
display(pd.DataFrame(report['folds'])[[
    'fold', 'training_rows', 'validation_rows', 'input_predictors',
    'encoded_predictors', 'training_statistics_verified',
    'state_unchanged_after_validation_transform', 'finite_outputs',
]])
unknown = pd.DataFrame({f['fold']: f['unknown_validation_counts'] for f in report['folds']})
unknown.columns.name = 'fold'
display(unknown.loc[unknown.sum(axis=1) > 0])
print('Evidencia: results/metrics/preprocessing_audit.json')


,fold,training_rows,validation_rows,input_predictors,encoded_predictors,training_statistics_verified,state_unchanged_after_validation_transform,finite_outputs
0,1,2831,708,18,213,True,True,True
1,2,2831,708,18,217,True,True,True
2,3,2831,708,18,214,True,True,True
3,4,2831,708,18,211,True,True,True
4,5,2832,707,18,210,True,True,True


fold,1,2,3,4,5
Application mode,1,0,1,0,1
Previous qualification,2,0,0,1,0
Nacionality,1,0,1,3,3
Mother's qualification,2,1,1,2,1
Father's qualification,2,3,3,2,3
Mother's occupation,0,1,0,2,2
Father's occupation,2,1,5,4,5


Evidencia: results/metrics/preprocessing_audit.json


## Conclusiones y limites

El preprocesamiento queda listo para integrarse dentro de un Pipeline con el clasificador, de modo que la validacion interna tambien lo ajuste solo con entrenamiento. No deben reutilizarse matrices transformadas con todo desarrollo para hacer validacion cruzada.

No se eliminan valores atipicos ni se incorporan variables nuevas. La mediana, la moda y el escalador no se recalculan al transformar validacion. El control de entrada es estructural: la validacion completa de rangos y codigos de la futura interfaz se implementara por separado.

El ejemplo reproduce el esquema historico. No se ha demostrado una mejora predictiva ni robustez ante todos los escenarios. Si una variable carece completamente de observaciones durante entrenamiento, requiere un protocolo especifico antes de admitir ese caso; la auditoria actual no simula ese entrenamiento.

La seleccion final del modelo, calibracion, equidad y evaluacion de la reserva siguen pendientes. Fuente y atribucion: `data/README.md`.
